# Populate World-Camera Calibration Data

This notebook is the reproducible entry point for populating the calibration assets under `data/` that support the world-camera calibration pipeline. It consolidates the important extraction logic that previously lived in scratch notebooks and documents the manually curated or MATLAB-generated assets alongside it.

Every write operation is disabled by default. Set the relevant `RUN_*` flag to `True` only after reviewing its source path. Existing output is preserved unless that section's `OVERWRITE_*` flag is also set to `True`.

The historical `data/xOld/` directory and generated implementation artifacts such as `data/__pycache__/` are intentionally outside the scope of this notebook.

## Contents and provenance

| Destination | Source | Operation |
| --- | --- | --- |
| `data/exampleWorldCameraImages/` | External archive of already selected examples | Validate and copy the curated indoor, outdoor, and planetarium TIFFs |
| `data/fisheyeLensCalibration/` | MATLAB Camera Calibrator archive | Copy the app session and checkerboard calibration images |
| `data/flatFieldingFunction/rawFrames/` | Fels Planetarium world-camera chunks | Build a temporary video and extract the documented time samples |
| `data/radiometricCorrectionRGB/rawFrames/` | Simultaneous cloudy-sky camera recording | Extract ten raw cloudy-sky frames used with the PR670 SPD |
| `data/darkNoise/` | Covered-camera recordings at fixed AGC states | Extract ten evenly spaced dark frames per state |
| `data/empircalAGC.mat` | Empirical indoor/outdoor recordings | Fit the AGC temporal model and export the selected calibration points |

The remaining root-level MAT files are documented in the final section because they originate from reference tables or MATLAB calibration analyses rather than from the Python extraction notebooks.

## Shared setup and write-safety helpers

The notebook may be launched from the repository root or from this notebook's directory. The setup cell locates the repository dynamically, adds the existing Python libraries to `sys.path`, and defines narrowly scoped helpers for preparing output directories.

In [ ]:
from __future__ import annotations

import importlib
import re
import shutil
import sys
import tempfile
from pathlib import Path

import cv2
import numpy as np


def locate_project_root(start: Path | None = None) -> Path:
    """Find the repository using directories that are stable across notebook launch locations."""
    start = (start or Path.cwd()).resolve()
    for candidate in (start, *start.parents):
        if (candidate / "data").is_dir() and (candidate / "code").is_dir():
            return candidate
    raise FileNotFoundError("Could not locate the lightLoggerAnalysis repository root.")


PROJECT_ROOT = locate_project_root()
DATA_ROOT = PROJECT_ROOT / "data"
MATLAB_IO_PYTHON = PROJECT_ROOT / "code" / "library" / "matlabIO" / "python_libraries"
SENSOR_UTILITY_PYTHON = PROJECT_ROOT / "code" / "library" / "sensor_utility"
FIT_AGC_PYTHON = PROJECT_ROOT / "code" / "fitAGCtoIlluminance"

for library_path in (MATLAB_IO_PYTHON, SENSOR_UTILITY_PYTHON, FIT_AGC_PYTHON):
    if str(library_path) not in sys.path:
        sys.path.insert(0, str(library_path))

def load_video_io():
    """Import the repository video utility only when an extraction section is run.

    video_io imports MATLAB's Python engine, so deferring this import lets the
    archive-copy and validation sections run in a plain Python kernel.
    """
    import video_io
    return video_io


print(f"Project root: {PROJECT_ROOT}")
print(f"Data root:    {DATA_ROOT}")

In [ ]:
def assert_safe_data_target(target: Path) -> Path:
    """Resolve a target and ensure it is below data/, never data/ itself."""
    resolved_target = target.resolve()
    resolved_data_root = DATA_ROOT.resolve()
    if resolved_target == resolved_data_root or resolved_data_root not in resolved_target.parents:
        raise ValueError(f"Refusing an unsafe output target: {resolved_target}")
    return resolved_target


def prepare_output_directory(target: Path, *, overwrite: bool) -> Path:
    """Create an empty data subdirectory, preserving existing contents unless explicitly allowed."""
    target = assert_safe_data_target(target)
    target.mkdir(parents=True, exist_ok=True)
    existing_items = list(target.iterdir())
    if existing_items and not overwrite:
        raise FileExistsError(
            f"{target} is not empty. Review it and set this section's OVERWRITE flag to True."
        )
    if overwrite:
        for item in existing_items:
            if item.is_dir():
                shutil.rmtree(item)
            else:
                item.unlink()
    return target


def copy_file(source: Path, destination: Path, *, overwrite: bool) -> None:
    """Copy one file while requiring explicit permission to replace an existing destination."""
    source = source.resolve()
    destination = assert_safe_data_target(destination)
    if not source.is_file():
        raise FileNotFoundError(source)
    if destination.exists() and not overwrite:
        raise FileExistsError(destination)
    destination.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(source, destination)


def write_tiff_stack(frames: np.ndarray | list[np.ndarray], output_dir: Path, *, overwrite: bool) -> None:
    """Write a frame stack as sequential, zero-based, uncompressed TIFF files."""
    output_dir = prepare_output_directory(output_dir, overwrite=overwrite)
    for frame_index, frame in enumerate(frames):
        output_path = output_dir / f"{frame_index}.tiff"
        written = cv2.imwrite(str(output_path), frame, [cv2.IMWRITE_TIFF_COMPRESSION, 1])
        if not written:
            raise IOError(f"OpenCV could not write {output_path}")
    print(f"Wrote {len(frames)} frames to {output_dir}")

# `data/exampleWorldCameraImages/`

**What this is:** A small curated gallery used to inspect representative world-camera images from indoor, outdoor, and planetarium conditions. The current archive contains ten indoor images, ten outdoor images, and nine planetarium images.

**Where it comes from:** The original selection notebook or exact source-frame indices could not be recovered from the repository. Consequently, this section does not pretend to regenerate the selection from raw recordings. It copies and validates an external archive whose files have already been named `indoor_N.tiff`, `outdoor_N.tiff`, and `planetarium_N.tiff`.

**What it does:** Validates the expected names, rejects unexpected files, and copies the curated TIFFs.

**Output:** `data/exampleWorldCameraImages/*.tiff`.

In [ ]:
RUN_EXAMPLE_IMAGE_COPY = False
OVERWRITE_EXAMPLE_IMAGES = False
EXAMPLE_IMAGE_ARCHIVE: Path | None = None  # Set to the external folder containing the curated TIFFs.
EXAMPLE_IMAGE_OUTPUT = DATA_ROOT / "exampleWorldCameraImages"
EXPECTED_EXAMPLE_COUNTS = {"indoor": 10, "outdoor": 10, "planetarium": 9}


def populate_example_world_camera_images(source_dir: Path, *, overwrite: bool = False) -> None:
    source_dir = source_dir.resolve()
    if source_dir == EXAMPLE_IMAGE_OUTPUT.resolve():
        raise ValueError("The example-image source and destination must be different directories.")

    expected_names = {
        f"{condition}_{index}.tiff"
        for condition, count in EXPECTED_EXAMPLE_COUNTS.items()
        for index in range(count)
    }
    actual_names = {path.name for path in source_dir.glob("*.tiff")}
    missing = sorted(expected_names - actual_names)
    unexpected = sorted(actual_names - expected_names)
    if missing or unexpected:
        raise ValueError(f"Example-image archive mismatch. Missing={missing}; unexpected={unexpected}")

    output_dir = prepare_output_directory(EXAMPLE_IMAGE_OUTPUT, overwrite=overwrite)
    for filename in sorted(expected_names):
        shutil.copy2(source_dir / filename, output_dir / filename)
    print(f"Copied {len(expected_names)} curated example images to {output_dir}")


if RUN_EXAMPLE_IMAGE_COPY:
    if EXAMPLE_IMAGE_ARCHIVE is None:
        raise ValueError("Set EXAMPLE_IMAGE_ARCHIVE before enabling this section.")
    populate_example_world_camera_images(EXAMPLE_IMAGE_ARCHIVE, overwrite=OVERWRITE_EXAMPLE_IMAGES)

# `data/fisheyeLensCalibration/`

**What this is:** The checkerboard images and MATLAB Single Camera Calibrator session used to estimate focal length, principal point, and fisheye distortion for the ArduCam B0392 IMX219 world camera.

**Where it comes from:** A manually collected checkerboard-calibration recording followed by interactive selection and fitting in MATLAB's Camera Calibrator app. The exact app session cannot be recreated faithfully by a Python notebook without repeating those interactive choices, so the session and accepted images are treated as a calibration archive.

**What it does:** Copies a validated archive containing `intrinsics_calibration_session.mat` and 47 sequential calibration TIFFs. The existing explanatory README in the destination is preserved.

**Output:** `data/fisheyeLensCalibration/intrinsics_calibration_session.mat` and `data/fisheyeLensCalibration/intrinsics_calibration_images/`. After population, open the session with `cameraCalibrator('intrinsics_calibration_session.mat')`; the exported intrinsics belong in `derived/arducamB0392cameraInstrinsics.mat`.

In [ ]:
RUN_FISHEYE_ARCHIVE_COPY = False
OVERWRITE_FISHEYE_ARCHIVE = False
FISHEYE_ARCHIVE_SOURCE: Path | None = None  # Folder containing the session MAT and image subdirectory.
FISHEYE_OUTPUT = DATA_ROOT / "fisheyeLensCalibration"


def populate_fisheye_calibration_archive(source_dir: Path, *, overwrite: bool = False) -> None:
    source_dir = source_dir.resolve()
    source_session = source_dir / "intrinsics_calibration_session.mat"
    source_images = source_dir / "intrinsics_calibration_images"
    expected_image_names = {f"{index}.tiff" for index in range(47)}
    actual_image_names = {path.name for path in source_images.glob("*.tiff")}

    if not source_session.is_file():
        raise FileNotFoundError(source_session)
    if actual_image_names != expected_image_names:
        raise ValueError(
            f"Expected 47 calibration images named 0.tiff through 46.tiff; found {len(actual_image_names)}."
        )

    FISHEYE_OUTPUT.mkdir(parents=True, exist_ok=True)
    copy_file(source_session, FISHEYE_OUTPUT / source_session.name, overwrite=overwrite)
    output_images = prepare_output_directory(
        FISHEYE_OUTPUT / "intrinsics_calibration_images", overwrite=overwrite
    )
    for filename in sorted(expected_image_names, key=lambda name: int(Path(name).stem)):
        shutil.copy2(source_images / filename, output_images / filename)
    print(f"Copied the fisheye calibration session and 47 images to {FISHEYE_OUTPUT}")


if RUN_FISHEYE_ARCHIVE_COPY:
    if FISHEYE_ARCHIVE_SOURCE is None:
        raise ValueError("Set FISHEYE_ARCHIVE_SOURCE before enabling this section.")
    populate_fisheye_calibration_archive(
        FISHEYE_ARCHIVE_SOURCE, overwrite=OVERWRITE_FISHEYE_ARCHIVE
    )

# `data/flatFieldingFunction/`

**What this is:** Raw Bayer frames used to estimate the spatial sensitivity imposed by the fisheye lens. The camera pointed at the nominally uniform Fels Planetarium dome and was rotated about its optical axis. Averaging frames across orientations reduces dome-specific spatial structure while preserving camera/lens structure.

**Where it comes from:** World-camera chunks stored in the lab Dropbox under `FLIC_admin/Equipment/ArduCam B0392 IMX219 Wide Angle M12/fielding_function/planetarium_fielding_function_raw`. These are the paths and selections recovered from `code/library/matlabIO/python_libraries/scratch2.ipynb`.

**What it does:** Converts the chunks to a temporary grayscale video with digital gain applied, looks up its actual frame rate, and extracts the documented samples for four rotation periods plus a 36-frame selection spread throughout the recording. Files are named sequentially in list order, not by original video-frame number.

**Output:** `data/flatFieldingFunction/rawFrames/{period1,period2,period3,period4,throughoutVideo}/`. The current MATLAB calibration uses `throughoutVideo`; the four period directories are retained for historical comparisons. `defineFlatFieldingFunction.m` linearizes and averages the frames, fits the flattened Gaussian, and writes `derived/flatFieldingFunction.mat`.

In [ ]:
RUN_FLAT_FIELD_EXTRACTION = False
OVERWRITE_FLAT_FIELD_FRAMES = False
FLAT_FIELD_RAW_CHUNKS = Path(
    "/Users/zacharykelly/Library/CloudStorage/Dropbox-Aguirre-BrainardLab/"
    "Zachary Kelly/FLIC_admin/Equipment/ArduCam B0392 IMX219 Wide Angle M12/"
    "fielding_function/planetarium_fielding_function_raw"
)
FLAT_FIELD_OUTPUT = DATA_ROOT / "flatFieldingFunction" / "rawFrames"

FLAT_FIELD_TIMES_SECONDS: dict[str, list[int]] = {
    "period1": [
        360, 361, 362, 363, 364, 365, 366, 367, 368, 369, 370, 371, 372, 373,
        374, 375, 376, 378, 384, 385, 386, 387, 388, 389, 390, 391, 392, 393,
        394, 395, 396, 397, 398, 399, 400, 401, 402, 403, 404, 405, 406, 407,
        408, 409,
    ],
    "period2": [
        480, 481, 482, 483, 484, 485, 486, 487, 488, 489, 490, 491, 492, 493,
        494, 495, 496, 497, 498, 499, 500, 501, 502, 503, 504, 505, 506, 510,
        511, 512, 513, 514, 515, 516, 517, 518, 519, 520, 521, 522, 523, 524,
        525, 526, 527, 528, 529, 530, 531, 532, 533, 534, 535, 536,
    ],
    "period3": [
        606, 607, 608, 609, 610, 611, 612, 613, 614, 615, 616, 617, 618, 619,
        620, 621, 622, 623, 624, 625, 626, 627, 628, 629, 630, 631, 632, 638,
        639, 640, 641, 642, 643, 644, 645, 646, 647, 648, 649, 650, 651, 652,
        653, 654, 655, 656, 657, 658, 659, 660, 661, 662, 663, 664,
    ],
    "period4": [
        734, 735, 736, 737, 738, 739, 740, 741, 742, 743, 744, 745, 746, 747,
        748, 749, 750, 751, 752, 753, 754, 755, 756, 757, 758, 759, 760, 766,
        767, 768, 769, 770, 771, 772, 773, 774, 775, 776, 777, 778, 779, 780,
        781, 782, 783, 784, 785, 786, 787, 788, 789, 790, 791, 792,
    ],
    "throughoutVideo": [
        366, 372, 393, 401, 427, 435, 456, 466, 488, 498, 518, 528, 550, 560,
        582, 592, 614, 624, 646, 656, 678, 686, 708, 718, 742, 752, 774, 784,
        806, 814, 836, 846, 868, 878, 912, 920,
    ],
}

In [ ]:
def populate_flat_field_frames(raw_chunks: Path, *, overwrite: bool = False) -> None:
    video_io = load_video_io()
    raw_chunks = raw_chunks.resolve()
    if not raw_chunks.is_dir():
        raise FileNotFoundError(raw_chunks)

    # The AVI is an intermediate only. Keeping it in a temporary directory avoids
    # creating a second large calibration artifact in either Dropbox or data/.
    with tempfile.TemporaryDirectory(prefix="populate_flat_field_") as temporary_dir:
        temporary_video = Path(temporary_dir) / "planetarium_fielding_function.avi"
        video_io.world_chunks_to_video(
            str(raw_chunks),
            output_path=str(temporary_video),
            verbose=True,
            convert_to_seconds=True,
            fill_missing_frames=True,
            apply_digital_gain=True,
        )
        frames_per_second = float(video_io.inspect_video_FPS(str(temporary_video)))
        print(f"Temporary flat-field video frame rate: {frames_per_second:g} fps")

        # Prepare the common root once so obsolete period directories do not survive
        # an intentional full regeneration.
        prepare_output_directory(FLAT_FIELD_OUTPUT, overwrite=overwrite)
        for period_name, times_seconds in FLAT_FIELD_TIMES_SECONDS.items():
            frame_indices = [round(time_seconds * frames_per_second) for time_seconds in times_seconds]
            frames = video_io.extract_frames_from_video(
                str(temporary_video), frame_indices, verbose=True, is_grayscale=True
            )
            write_tiff_stack(frames, FLAT_FIELD_OUTPUT / period_name, overwrite=False)


if RUN_FLAT_FIELD_EXTRACTION:
    populate_flat_field_frames(
        FLAT_FIELD_RAW_CHUNKS, overwrite=OVERWRITE_FLAT_FIELD_FRAMES
    )

# `data/radiometricCorrectionRGB/`

**What this is:** A paired calibration between the spectral radiance of a cloudy sky measured with a PR670 and raw IMX219 images of that same sky. It is used to derive multiplicative RGB radiometric weights.

**Where it comes from:** The world-camera chunks are in Dropbox under `FLIC_data/LightLoggerRadCal/W1P1M1/radiometricCorrectionRGB/cloudyDayRecording`. The notebook logic comes from `code/preprocessRecordingData/another_scratch.ipynb`. The PR670 file `CloudySkySPD_37degSolarElevation.mat` and the illustrative `cropExample.tiff` are separately archived measurement inputs; this notebook does not recreate them.

**What it does:** Builds a temporary video without digital-gain, response-linearization, or color-weight corrections, then extracts grayscale frames 8000 through 8009. Preserving the uncorrected sensor values is essential because the downstream calibration is estimating those corrections.

**Output:** `data/radiometricCorrectionRGB/rawFrames/0.tiff` through `9.tiff`. `defineRadiometricWeights.m` combines these frames with the PR670 SPD and writes `derived/radiometricCorrectionRGB.mat`.

In [ ]:
RUN_RADIOMETRIC_FRAME_EXTRACTION = False
OVERWRITE_RADIOMETRIC_FRAMES = False
RADIOMETRIC_RAW_CHUNKS = Path(
    "/Users/zacharykelly/Library/CloudStorage/Dropbox-Aguirre-BrainardLab/"
    "Zachary Kelly/FLIC_data/LightLoggerRadCal/W1P1M1/"
    "radiometricCorrectionRGB/cloudyDayRecording"
)
RADIOMETRIC_OUTPUT = DATA_ROOT / "radiometricCorrectionRGB" / "rawFrames"
RADIOMETRIC_FRAME_INDICES = list(range(8000, 8010))


def populate_radiometric_frames(raw_chunks: Path, *, overwrite: bool = False) -> None:
    video_io = load_video_io()
    raw_chunks = raw_chunks.resolve()
    if not raw_chunks.is_dir():
        raise FileNotFoundError(raw_chunks)

    with tempfile.TemporaryDirectory(prefix="populate_radiometric_") as temporary_dir:
        temporary_video = Path(temporary_dir) / "cloudy_day_recording.avi"
        video_io.world_chunks_to_video(
            str(raw_chunks),
            str(temporary_video),
            apply_digital_gain=False,
            convert_to_seconds=True,
            fill_missing_frames=True,
            verbose=True,
            linearize_camera_responsivity=False,
            apply_color_weights=False,
        )
        frames = video_io.extract_frames_from_video(
            str(temporary_video), RADIOMETRIC_FRAME_INDICES, is_grayscale=True
        )
        write_tiff_stack(frames, RADIOMETRIC_OUTPUT, overwrite=overwrite)


if RUN_RADIOMETRIC_FRAME_EXTRACTION:
    populate_radiometric_frames(
        RADIOMETRIC_RAW_CHUNKS, overwrite=OVERWRITE_RADIOMETRIC_FRAMES
    )

# `data/darkNoise/`

**What this is:** Raw Bayer dark frames acquired with the lens cap installed, the camera wrapped in a black shroud, and the room dark. Separate recordings cover five fixed AGC states because exposure and analog gain can change the camera's dark behavior.

**Where it comes from:** The canonical recordings are documented in Dropbox at `FLIC_admin/Equipment/ArduCam B0392 IMX219 Wide Angle M12/darkNoiseCalibrations`. The recovered scratch notebook read a mounted working copy from `/Volumes/T7 Shield/darkNoise`; that remains the default below and can be changed to the canonical archive. Each child directory must be one raw world-camera chunk recording and should retain its metadata-rich `AGCstate*_AGain-*_DGain-*_E-*` name.

**What it does:** Converts each state recording to a temporary uncorrected grayscale video, chooses ten indices evenly across its full duration, and writes the selected dark frames. Digital gain, response linearization, and color weighting remain disabled so the TIFFs preserve the calibration measurement.

**Output:** One ten-frame directory per state beneath `data/darkNoise/`. The README is preserved. `defineDarkSignal.m` subsequently computes the median dark signal and writes `derived/darkSignal.mat`.

In [ ]:
RUN_DARK_NOISE_EXTRACTION = False
OVERWRITE_DARK_NOISE_FRAMES = False
DARK_NOISE_RAW_ROOT = Path("/Volumes/T7 Shield/darkNoise")
DARK_NOISE_OUTPUT = DATA_ROOT / "darkNoise"
DARK_NOISE_FRAME_COUNT = 10
DARK_NOISE_STATE_PATTERN = re.compile(r"^AGCstate[1-5](?:_|$)")


def populate_dark_noise_frames(raw_root: Path, *, overwrite: bool = False) -> None:
    video_io = load_video_io()
    raw_root = raw_root.resolve()
    if not raw_root.is_dir():
        raise FileNotFoundError(raw_root)

    state_recordings = sorted(
        path
        for path in raw_root.iterdir()
        if path.is_dir() and DARK_NOISE_STATE_PATTERN.match(path.name)
    )
    if len(state_recordings) != 5:
        raise ValueError(
            f"Expected five AGC-state recording directories under {raw_root}; found {len(state_recordings)}."
        )

    for recording_path in state_recordings:
        with tempfile.TemporaryDirectory(prefix=f"populate_{recording_path.name}_") as temporary_dir:
            temporary_video = Path(temporary_dir) / f"{recording_path.name}.avi"
            video_io.world_chunks_to_video(
                str(recording_path),
                str(temporary_video),
                apply_digital_gain=False,
                convert_to_seconds=True,
                fill_missing_frames=False,
                verbose=True,
                linearize_camera_responsivity=False,
                apply_color_weights=False,
            )
            video_frame_count = int(video_io.inspect_video_frame_count(str(temporary_video)))
            frame_indices = np.linspace(
                0, video_frame_count - 1, DARK_NOISE_FRAME_COUNT, dtype=np.uint64
            ).tolist()
            frames = video_io.extract_frames_from_video(
                str(temporary_video), frame_indices, is_grayscale=True
            )
            write_tiff_stack(
                frames, DARK_NOISE_OUTPUT / recording_path.name, overwrite=overwrite
            )


if RUN_DARK_NOISE_EXTRACTION:
    populate_dark_noise_frames(
        DARK_NOISE_RAW_ROOT, overwrite=OVERWRITE_DARK_NOISE_FRAMES
    )

# Root-level files in `data/`

The top level of `data/` contains several MAT files rather than another directory. Only the AGC-to-illuminance file came from one of the consolidated Python notebooks.

## `empircalAGC.mat`

**Source:** Raw `GKA` recordings from the 2026 scripted indoor/outdoor dataset mounted at `/Volumes/FLIC_raw/NEWscriptedIndoorOutdoorVideos2026`.

**Operation:** Run the empirical AGC temporal-kernel analysis, cache saturation diagnostics, select points below the configured saturation limit and above the model-correlation threshold, and write the linear-scale camera-AGC/illuminance point cloud. This is the key workflow from `fit_agc_to_illuminance_processing.ipynb`.

**Output:** `deriveEmpircalAGCAndIlluminance.py` writes `data/empircalAGC.mat` directly. The file contains the MATLAB struct `empiralAGC` with the fields `cameraScoreLinear` and `msIlluminance`. The later MATLAB piecewise log-log fit is model fitting, not data population, so it is not run here.

In [ ]:
RUN_AGC_TO_ILLUMINANCE = False
OVERWRITE_AGC_TO_ILLUMINANCE = False
AGC_RAW_ROOT = Path("/Volumes/FLIC_raw/NEWscriptedIndoorOutdoorVideos2026")
AGC_MAXIMUM_SUBJECTS = 4
AGC_SUBJECTS_TO_SKIP = {"FLIC_18"}
AGC_MAXIMUM_SATURATION_PERCENT = 40.0
AGC_INITIAL_SAMPLES_TO_EXCLUDE = 100
AGC_DATA_OUTPUT = DATA_ROOT / "empircalAGC.mat"


def natural_sort_key(path_or_name: Path | str) -> list[int | str]:
    return [int(piece) if piece.isdigit() else piece.lower() for piece in re.split(r"(\d+)", str(path_or_name))]


def populate_agc_to_illuminance(raw_root: Path, *, overwrite: bool = False) -> None:
    raw_root = raw_root.resolve()
    if not raw_root.is_dir():
        raise FileNotFoundError(raw_root)

    derive_module_dir = PROJECT_ROOT / "code" / "defineWorldCameraCalibration"
    if str(derive_module_dir) not in sys.path:
        sys.path.insert(0, str(derive_module_dir))
    import deriveEmpircalAGCAndIlluminance
    importlib.reload(deriveEmpircalAGCAndIlluminance)

    recording_paths: list[str] = []
    valid_subject_count = 0
    for subject_dir in sorted(raw_root.iterdir(), key=natural_sort_key):
        if valid_subject_count >= AGC_MAXIMUM_SUBJECTS:
            break
        if (
            not subject_dir.is_dir()
            or subject_dir.name.startswith(".")
            or subject_dir.name in AGC_SUBJECTS_TO_SKIP
        ):
            continue
        for activity_dir in sorted(subject_dir.iterdir(), key=natural_sort_key):
            if not activity_dir.is_dir() or activity_dir.name.startswith("."):
                continue
            recording_path = activity_dir / "GKA"
            if not recording_path.is_dir():
                raise FileNotFoundError(recording_path)
            recording_paths.append(str(recording_path))
        valid_subject_count += 1

    if AGC_DATA_OUTPUT.exists() and not overwrite:
        raise FileExistsError(AGC_DATA_OUTPUT)

    deriveEmpircalAGCAndIlluminance.derive_empircal_agc_and_illuminance(
        recording_paths,
        output_path=AGC_DATA_OUTPUT,
        maximum_saturation_percent=AGC_MAXIMUM_SATURATION_PERCENT,
        initial_samples_to_exclude=AGC_INITIAL_SAMPLES_TO_EXCLUDE,
        illuminance_diagnostics=True,
    )
    print(f"Generated AGC-to-illuminance data at {AGC_DATA_OUTPUT}")


if RUN_AGC_TO_ILLUMINANCE:
    populate_agc_to_illuminance(
        AGC_RAW_ROOT, overwrite=OVERWRITE_AGC_TO_ILLUMINANCE
    )

## Other root-level reference and calibration MAT files

- `ASM7341_spectralSensitivity.mat` is a reference table transcribed from the manufacturer-supplied `AS7341_Filter_Templates.xlsx` spreadsheet.
- `IMX219_spectralSensitivity.mat` is a reference table corresponding to Figure 18 of Pagnutti et al. (2017), supplied by the paper's first author.
- `camera_linearity_ND0_ND0p4_rgb_means.mat` is produced by the MATLAB function `analyze_camera_linearity_data.m` from collected integrating-sphere calibration measurements. It is saved in MATLAB's current directory and should be reviewed before being placed in `data/`.

These files should remain curated calibration inputs rather than being silently overwritten by this notebook.

# Validate the populated data tree

This read-only audit checks the expected counts and key files after any population sections have run. It deliberately ignores `xOld`, hidden Finder metadata, and Python cache files.

In [ ]:
def numbered_tiff_count(directory: Path) -> int:
    return len([path for path in directory.glob("*.tiff") if path.stem.isdigit()])


checks = {
    "example indoor images": len(list((DATA_ROOT / "exampleWorldCameraImages").glob("indoor_*.tiff"))) == 10,
    "example outdoor images": len(list((DATA_ROOT / "exampleWorldCameraImages").glob("outdoor_*.tiff"))) == 10,
    "example planetarium images": len(list((DATA_ROOT / "exampleWorldCameraImages").glob("planetarium_*.tiff"))) == 9,
    "fisheye session": (DATA_ROOT / "fisheyeLensCalibration" / "intrinsics_calibration_session.mat").is_file(),
    "fisheye images": numbered_tiff_count(DATA_ROOT / "fisheyeLensCalibration" / "intrinsics_calibration_images") == 47,
    "flat-field period1": numbered_tiff_count(FLAT_FIELD_OUTPUT / "period1") == 44,
    "flat-field period2": numbered_tiff_count(FLAT_FIELD_OUTPUT / "period2") == 54,
    "flat-field period3": numbered_tiff_count(FLAT_FIELD_OUTPUT / "period3") == 54,
    "flat-field period4": numbered_tiff_count(FLAT_FIELD_OUTPUT / "period4") == 54,
    "flat-field throughoutVideo": numbered_tiff_count(FLAT_FIELD_OUTPUT / "throughoutVideo") == 36,
    "radiometric frames": numbered_tiff_count(RADIOMETRIC_OUTPUT) == 10,
    "five dark-noise states": len([path for path in DARK_NOISE_OUTPUT.iterdir() if path.is_dir() and DARK_NOISE_STATE_PATTERN.match(path.name)]) == 5,
    "ten frames in every dark-noise state": all(
        numbered_tiff_count(path) == 10
        for path in DARK_NOISE_OUTPUT.iterdir()
        if path.is_dir() and DARK_NOISE_STATE_PATTERN.match(path.name)
    ),
    "AGC-to-illuminance MAT": AGC_DATA_OUTPUT.is_file(),
    "AS7341 sensitivity MAT": (DATA_ROOT / "ASM7341_spectralSensitivity.mat").is_file(),
    "IMX219 sensitivity MAT": (DATA_ROOT / "IMX219_spectralSensitivity.mat").is_file(),
    "camera-linearity MAT": (DATA_ROOT / "camera_linearity_ND0_ND0p4_rgb_means.mat").is_file(),
}

for description, passed in checks.items():
    print(f"{'PASS' if passed else 'FAIL'}  {description}")

if not all(checks.values()):
    raise AssertionError("One or more expected calibration-data checks failed.")